# 13 — App Log Analytics Views

**Jackson and Jackson HR Digital**

Databricks Apps exports the Hire Right app's runtime logs to Unity Catalog as an OpenTelemetry table
(`<prefix>_otel_logs`). Each row is one log line, but the meaning is buried in a `VARIANT` `attributes` map and a
free-text `body`. The key field is `attributes:['app.log_source']`, which says which process wrote the line:

| Source | What it is | What we pull out of it |
|---|---|---|
| `BUILD` | Deployment / build process | deployment ID, start command, deploy duration, success/failure, error |
| `HTTP` | Databricks Apps proxy access log (1 line per user request) | method, route, feature, status, bytes, user ID, browser, approx duration |
| `APP` | The app's own stdout (uvicorn + Python logging) | warnings, errors, tracebacks, outbound calls to the agent endpoint |
| *(no source, `event.name` set)* | Platform system events, e.g. `app.auth` logins | user email, outcome, reason |

This notebook publishes five flat, fully-commented views that Genie and AI/BI dashboards can query directly.

| View | Grain | Answers |
|---|---|---|
| `v_app_log_events` | 1 row per log line | errors / warnings, timelines, log volume by source |
| `v_app_http_requests` | 1 row per user HTTP request | feature usage, error rates, slow requests, candidates viewed, who uses the app |
| `v_app_outbound_calls` | 1 row per call the app made to a dependency | can the app reach the agent endpoint? 403 / 5xx failures |
| `v_app_deployments` | 1 row per deployment | deploy history, failures and why, start command, errors per deploy |
| `v_app_auth_events` | 1 row per sign-in / session event | who signed in, failed logins |

### Prerequisites
- App telemetry is enabled on the app (`telemetry_export_destinations` in `databricks.yml`) and the app has been
  deployed at least once since, so `<prefix>_otel_logs` exists and has rows

**Next notebook:** `14_create_app_log_genie_agent.ipynb`


## Setup

In [ ]:
dbutils.widgets.text("app_log_catalog",      "bldemos",                    "UC Catalog (app telemetry)")
dbutils.widgets.text("app_log_schema",       "hrd_2030_app_observability", "UC Schema (app telemetry)")
dbutils.widgets.text("app_log_table_prefix", "hrd_app",                    "Telemetry Table Prefix")

In [ ]:
import os

app_log_catalog      = dbutils.widgets.get("app_log_catalog")      or os.getenv("APP_LOG_CATALOG", "bldemos")
app_log_schema       = dbutils.widgets.get("app_log_schema")       or os.getenv("APP_LOG_SCHEMA", "hrd_2030_app_observability")
app_log_table_prefix = dbutils.widgets.get("app_log_table_prefix") or os.getenv("APP_LOG_TABLE_PREFIX", "hrd_app")

target = f"{app_log_catalog}.{app_log_schema}"
logs   = f"{target}.{app_log_table_prefix}_otel_logs"

print(f"Source logs  : {logs}")
print(f"Views schema : {target}")

# On a fresh deploy the telemetry table appears only after the app's first log lines are exported
import time
for _ in range(20):
    if spark.catalog.tableExists(logs):
        break
    print(f"Waiting for {logs} to be created by app telemetry…")
    time.sleep(15)
else:
    dbutils.notebook.exit(f"SKIPPED: {logs} does not exist — enable app telemetry and redeploy the app first")

## View Definitions

Each view is declared with a table comment and a comment on every column — Genie uses these
descriptions to map business language ("failed deploy", "most used feature") onto the right columns.
The regexes parse the three `body` formats (proxy access log, uvicorn/Python log, platform BUILD log).

In [ ]:
BASE = f"""
  SELECT l.record_id,
     l.time,
     COALESCE(NULLIF(l.service_name, ''), l.resource.attributes:['app.name']::string) AS app_name,
     CASE WHEN l.attributes:['event.name'] IS NOT NULL THEN 'SYSTEM'
          ELSE NULLIF(l.attributes:['app.log_source']::string, '') END    AS log_source,
     NULLIF(l.attributes:['app.deployment_id']::string, '')               AS deployment_id,
     l.attributes:['event.name']::string                                  AS system_event,
     NULLIF(l.severity_text, '')                                          AS raw_severity,
     COALESCE(l.body::string, '')                                         AS body_text
  FROM {logs} l
"""

VIEWS = []

# ── v_app_log_events ─────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_app_log_events",
    comment=(
        "One row per Hire Right app log line (OpenTelemetry logs exported by Databricks Apps). "
        "Flattens the nested attributes/body into plain columns: log source, normalized severity, deployment, "
        "message category, and a clean one-line message. Use for error/warning hunting, log volume, and timelines."
    ),
    columns=[
        ("record_id",        "Unique ID of the log record."),
        ("event_time",       "Timestamp (UTC) the log line was written."),
        ("event_date",       "Calendar date (UTC) of event_time. Use for daily trends."),
        ("event_hour",       "event_time truncated to the hour (UTC). Use for hourly activity."),
        ("app_name",         "Databricks App that produced the log (hire-right-agent-app)."),
        ("log_source",       "Where the line came from: BUILD = deployment/build process, APP = the app's own stdout/stderr (uvicorn + Python logging), HTTP = Databricks Apps proxy access log (one line per browser/API request), SYSTEM = platform event such as a user login."),
        ("deployment_id",    "App deployment the line belongs to. Join key to v_app_deployments. NULL for SYSTEM events."),
        ("severity",         "Normalized severity: ERROR, WARN, or INFO. Derived from the raw OTel severity, Python log prefixes (WARNING:/ERROR:), tracebacks, and HTTP status codes (5xx = ERROR, 4xx = WARN)."),
        ("is_error",         "True when severity = ERROR."),
        ("is_warning",       "True when severity = WARN."),
        ("message_category", "What kind of line this is: 'Inbound request' (proxy access log — count requests with these), 'Request echo' (uvicorn's copy of the same request — do not count), 'Outbound call', 'Deploy step', 'Deploy error', 'Server lifecycle', 'Login / auth', 'App warning', 'App error', or 'App log'."),
        ("logger_name",      "Python logger that wrote the line (e.g. db, httpx, app) when the line uses the 'LEVEL:logger:message' format; otherwise NULL."),
        ("message",          "First line of the log message with level prefixes removed — the human-readable summary."),
        ("message_full",     "Complete raw log message, including multi-line tracebacks and SQL."),
    ],
    sql=f"""
WITH base AS ({BASE}),
classified AS (
  SELECT *,
     CASE
       WHEN log_source = 'SYSTEM' THEN COALESCE(raw_severity, 'INFO')
       WHEN log_source = 'HTTP' THEN
         CASE WHEN regexp_extract(body_text, '" (\\\\d{{3}}) ', 1) >= '500' THEN 'ERROR'
              WHEN regexp_extract(body_text, '" (\\\\d{{3}}) ', 1) >= '400' THEN 'WARN'
              ELSE 'INFO' END
       WHEN raw_severity = 'ERROR' OR body_text RLIKE '^(ERROR|CRITICAL):' OR body_text LIKE 'Traceback%' THEN 'ERROR'
       WHEN body_text RLIKE '"HTTP/1\\\\.[01] 5\\\\d\\\\d' THEN 'ERROR'
       WHEN raw_severity IN ('WARN', 'WARNING') OR body_text RLIKE '^WARNING:' OR body_text RLIKE 'Warning:' THEN 'WARN'
       WHEN body_text RLIKE '"HTTP/1\\\\.[01] 4\\\\d\\\\d' THEN 'WARN'
       ELSE 'INFO'
     END AS severity
  FROM base
)
SELECT record_id,
   time                              AS event_time,
   to_date(time)                     AS event_date,
   date_trunc('HOUR', time)          AS event_hour,
   app_name,
   log_source,
   deployment_id,
   severity,
   severity = 'ERROR'                AS is_error,
   severity = 'WARN'                 AS is_warning,
   CASE
     WHEN log_source = 'SYSTEM' THEN 'Login / auth'
     WHEN log_source = 'HTTP' THEN 'Inbound request'
     WHEN log_source = 'BUILD' AND severity = 'ERROR' THEN 'Deploy error'
     WHEN log_source = 'BUILD' THEN 'Deploy step'
     WHEN body_text RLIKE '^INFO:httpx:HTTP Request:' THEN 'Outbound call'
     WHEN body_text RLIKE '^INFO: +\\\\S+ - "' THEN 'Request echo'
     WHEN body_text RLIKE '^INFO: +(Started server|Waiting for application|Application (startup|shutdown)|Uvicorn running|Shutting down|Finished server)' THEN 'Server lifecycle'
     WHEN severity = 'ERROR' THEN 'App error'
     WHEN severity = 'WARN' THEN 'App warning'
     ELSE 'App log'
   END                               AS message_category,
   NULLIF(regexp_extract(body_text, '^(?:DEBUG|INFO|WARNING|ERROR|CRITICAL):([A-Za-z0-9_.]+):', 1), '') AS logger_name,
   CASE
     WHEN log_source = 'SYSTEM' THEN concat('User login/session event: ', COALESCE(system_event, ''))
     ELSE trim(regexp_replace(
            regexp_replace(split(body_text, '\\n')[0], '^\\\\[(INFO|ERROR|WARN|WARNING)\\\\] ', ''),
            '^(DEBUG|INFO|WARNING|ERROR|CRITICAL):([A-Za-z0-9_.]+:)? *', ''))
   END                               AS message,
   body_text                         AS message_full
FROM classified
""",
))

# ── v_app_http_requests ──────────────────────────────────────────────────
HTTP_RE = '^(\\\\S+) - (\\\\S+) \\\\[([^\\\\]]+)\\\\] "(\\\\S+) (\\\\S+) [^"]*" (\\\\d{3}) (\\\\S+) "([^"]*)" "([^"]*)"'
VIEWS.append(dict(
    name="v_app_http_requests",
    comment=(
        "One row per HTTP request a user (browser or API client) made to the Hire Right app, parsed from the "
        "Databricks Apps proxy access log (log_source = HTTP). Use for feature usage, endpoint popularity, error "
        "rates, approximate response times, which candidates were viewed, and who is using the app."
    ),
    columns=[
        ("record_id",               "Unique ID of the underlying log record."),
        ("request_time",            "Timestamp (UTC) the request arrived at the app proxy."),
        ("request_date",            "Calendar date (UTC) of request_time."),
        ("request_hour",            "request_time truncated to the hour (UTC). Use for busiest-hour questions."),
        ("deployment_id",           "App deployment that served the request. Join key to v_app_deployments."),
        ("http_method",             "HTTP verb: GET (read/view) or POST/DELETE (actions)."),
        ("path",                    "Exact URL path requested, without query string (e.g. /api/candidates/C002/annotations)."),
        ("route",                   "Path with IDs replaced by placeholders (e.g. /api/candidates/{candidate_id}/annotations). Group by this to count endpoint usage."),
        ("app_feature",             "Business feature the route belongs to: Candidate list, HR notes, Resume PDF, Agent chat, Genie analytics, Offer letter, Jobs, App page load, Health check, or Other."),
        ("candidate_id",            "Candidate ID in the URL (e.g. C002) when the request was about one candidate; NULL otherwise."),
        ("status_code",             "HTTP response status code (200, 403, 500, …)."),
        ("status_class",            "Status bucket: 2xx Success, 3xx Redirect, 4xx Client error, 5xx Server error."),
        ("is_error",                "True for 4xx and 5xx responses."),
        ("response_bytes",          "Size of the response body in bytes."),
        ("approx_duration_seconds", "Approximate request duration in seconds: time the proxy logged the response minus the request arrival time. 1-second resolution — use for spotting slow requests (e.g. long agent chats), not precise latency."),
        ("user_id",                 "Databricks numeric user ID of the signed-in user who made the request; NULL for unauthenticated redirects."),
        ("workspace_id",            "Databricks workspace ID the user signed in through."),
        ("client_ip",               "Public IP address of the user's browser."),
        ("browser",                 "Browser family parsed from the user agent: Chrome, Safari, Firefox, Edge, or Other."),
        ("user_agent",              "Full raw User-Agent header."),
    ],
    sql=f"""
WITH parsed AS (
  SELECT l.record_id,
     l.time AS log_time,
     l.attributes:['app.deployment_id']::string AS deployment_id,
     l.body::string AS b
  FROM {logs} l
  WHERE l.attributes:['app.log_source']::string = 'HTTP'
),
fields AS (
  SELECT record_id, log_time, deployment_id,
     regexp_extract(b, '{HTTP_RE}', 1) AS client_ip,
     regexp_extract(b, '{HTTP_RE}', 2) AS who,
     regexp_extract(b, '{HTTP_RE}', 3) AS ts_text,
     regexp_extract(b, '{HTTP_RE}', 4) AS http_method,
     regexp_replace(regexp_extract(b, '{HTTP_RE}', 5), '\\\\?.*$', '') AS path,
     CAST(regexp_extract(b, '{HTTP_RE}', 6) AS INT) AS status_code,
     TRY_CAST(regexp_extract(b, '{HTTP_RE}', 7) AS BIGINT) AS response_bytes,
     regexp_extract(b, '{HTTP_RE}', 9) AS user_agent
  FROM parsed
),
routed AS (
  SELECT *,
     to_timestamp(ts_text, 'dd/MMM/yyyy:HH:mm:ss Z') AS request_time,
     regexp_replace(regexp_replace(path, '/C[0-9]+(/|$)', '/{{candidate_id}}$1'), '/[0-9]+(/|$)', '/{{id}}$1') AS route
  FROM fields
)
SELECT record_id,
   request_time,
   to_date(request_time)                     AS request_date,
   date_trunc('HOUR', request_time)          AS request_hour,
   deployment_id,
   http_method,
   path,
   route,
   CASE
     WHEN route = '/api/candidates'                      THEN 'Candidate list'
     WHEN route LIKE '/api/candidates/%/annotations%'    THEN 'HR notes'
     WHEN route LIKE '/api/resume-pdf/%'                 THEN 'Resume PDF'
     WHEN route LIKE '/api/chat%'                        THEN 'Agent chat'
     WHEN route = '/api/genie'                           THEN 'Genie analytics'
     WHEN route LIKE '/api/offer-letter/%'               THEN 'Offer letter'
     WHEN route = '/api/jobs'                            THEN 'Jobs'
     WHEN route = '/api/health'                          THEN 'Health check'
     WHEN route NOT LIKE '/api/%'                        THEN 'App page load'
     ELSE 'Other'
   END                                       AS app_feature,
   NULLIF(regexp_extract(path, '/(C[0-9]+)(/|$)', 1), '') AS candidate_id,
   status_code,
   CASE WHEN status_code >= 500 THEN '5xx Server error'
        WHEN status_code >= 400 THEN '4xx Client error'
        WHEN status_code >= 300 THEN '3xx Redirect'
        ELSE '2xx Success' END               AS status_class,
   status_code >= 400                        AS is_error,
   response_bytes,
   GREATEST(unix_timestamp(log_time) - unix_timestamp(request_time), 0) AS approx_duration_seconds,
   CASE WHEN who = '-' THEN NULL ELSE split(who, '@')[0] END AS user_id,
   CASE WHEN who = '-' THEN NULL ELSE split(who, '@')[1] END AS workspace_id,
   client_ip,
   CASE WHEN user_agent LIKE '%Edg/%'     THEN 'Edge'
        WHEN user_agent LIKE '%Firefox/%' THEN 'Firefox'
        WHEN user_agent LIKE '%Chrome/%'  THEN 'Chrome'
        WHEN user_agent LIKE '%Safari/%'  THEN 'Safari'
        ELSE 'Other' END                     AS browser,
   user_agent
FROM routed
""",
))

# ── v_app_outbound_calls ─────────────────────────────────────────────────
OUT_RE = '^INFO:httpx:HTTP Request: (\\\\S+) (\\\\S+) "HTTP/[0-9.]+ ([0-9]{3}) ?([^"]*)"'
VIEWS.append(dict(
    name="v_app_outbound_calls",
    comment=(
        "One row per outbound HTTP call the Hire Right app made to another service (logged by the Python httpx "
        "client) — chiefly the Hire Right agent serving endpoint. Use to check whether the app can reach its "
        "dependencies and to find permission (403) or server (5xx) failures that the user-facing request log hides."
    ),
    columns=[
        ("record_id",      "Unique ID of the underlying log record."),
        ("call_time",      "Timestamp (UTC) the outbound call completed."),
        ("call_date",      "Calendar date (UTC) of call_time."),
        ("deployment_id",  "App deployment that made the call. Join key to v_app_deployments."),
        ("http_method",    "HTTP verb of the outbound call (usually POST)."),
        ("target_host",    "Host name that was called."),
        ("target_service", "Friendly name of the dependency, e.g. 'Serving endpoint: hire-right-agent-endpoint'."),
        ("url",            "Full URL that was called."),
        ("status_code",    "HTTP status the dependency returned."),
        ("status_reason",  "Status text the dependency returned (OK, Forbidden, …)."),
        ("is_error",       "True when the dependency returned 4xx or 5xx."),
    ],
    sql=f"""
WITH m AS (
  SELECT l.record_id, l.time, l.attributes:['app.deployment_id']::string AS deployment_id, l.body::string AS b
  FROM {logs} l
  WHERE l.attributes:['app.log_source']::string = 'APP'
AND l.body::string LIKE 'INFO:httpx:HTTP Request:%'
),
f AS (
  SELECT record_id, time, deployment_id,
     regexp_extract(b, '{OUT_RE}', 1) AS http_method,
     regexp_extract(b, '{OUT_RE}', 2) AS url,
     CAST(regexp_extract(b, '{OUT_RE}', 3) AS INT) AS status_code,
     regexp_extract(b, '{OUT_RE}', 4) AS status_reason
  FROM m
)
SELECT record_id,
   time                     AS call_time,
   to_date(time)            AS call_date,
   deployment_id,
   http_method,
   parse_url(url, 'HOST')   AS target_host,
   CASE WHEN url RLIKE '/serving-endpoints/[^/]+/'
          THEN concat('Serving endpoint: ', regexp_extract(url, '/serving-endpoints/([^/]+)/', 1))
        WHEN url LIKE '%/api/2.0/genie/%' THEN 'Genie API'
        WHEN url LIKE '%/api/2.0/sql/%'   THEN 'SQL warehouse API'
        ELSE parse_url(url, 'HOST') END AS target_service,
   url,
   status_code,
   status_reason,
   status_code >= 400       AS is_error
FROM f
""",
))

# ── v_app_deployments ────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_app_deployments",
    comment=(
        "One row per Hire Right app deployment, rolled up from BUILD log lines. Use for deploy history, "
        "success/failure, why a deployment failed, how long deploys take, which start command was used, and how "
        "many warnings/errors the app logged while that deployment was running."
    ),
    columns=[
        ("deployment_id",          "Unique app deployment ID. Join key to the other v_app_* views."),
        ("started_at",             "Timestamp (UTC) the deployment started."),
        ("ended_at",               "Timestamp (UTC) of the last BUILD line for the deployment."),
        ("deploy_date",            "Calendar date (UTC) of started_at."),
        ("outcome",                "Succeeded, Failed, or In progress (no final BUILD line yet)."),
        ("deploy_seconds",         "Build-to-running duration in seconds as reported by the platform ('Deployment … ended in Xs')."),
        ("start_command",          "Command the platform used to start the app. 'python app.py' means app.yaml/bundle config was not applied."),
        ("requirements_installed", "True if pip packages were (re)installed; false when requirements were unchanged and installation was skipped."),
        ("files_updated",          "Number of source files that changed in this deployment."),
        ("failure_reason",         "First BUILD error message for failed deployments; NULL if the deployment succeeded."),
        ("app_warnings",           "Number of WARN lines the app logged while this deployment was live."),
        ("app_errors",             "Number of ERROR lines the app logged while this deployment was live (excluding BUILD errors)."),
        ("http_requests",          "Number of user HTTP requests served by this deployment."),
    ],
    sql="""
WITH build AS (
  SELECT * FROM __TARGET__.v_app_log_events WHERE log_source = 'BUILD'
),
roll AS (
  SELECT deployment_id,
     MIN(event_time) AS started_at,
     MAX(event_time) AS ended_at,
     MAX(TRY_CAST(regexp_extract(message, 'ended in ([0-9.]+)s', 1) AS DOUBLE)) AS deploy_seconds,
     MAX(NULLIF(regexp_extract(message, 'Starting app with command: \\\\[(.*)\\\\]', 1), '')) AS start_command,
     COUNT_IF(message LIKE 'Requirements installed%') > 0 AS requirements_installed,
     COUNT_IF(message LIKE 'Updated file:%') AS files_updated,
     COUNT_IF(message = 'Deployment successful') > 0 AS succeeded,
     MIN_BY(message, CASE WHEN is_error THEN event_time END) AS failure_reason,
     COUNT_IF(is_error) AS build_errors
  FROM build
  GROUP BY deployment_id
),
app AS (
  SELECT deployment_id,
     COUNT_IF(is_warning) AS app_warnings,
     COUNT_IF(is_error)   AS app_errors,
     COUNT_IF(log_source = 'HTTP') AS http_requests
  FROM __TARGET__.v_app_log_events
  WHERE log_source IN ('APP', 'HTTP')
  GROUP BY deployment_id
)
SELECT roll.deployment_id,
   roll.started_at,
   roll.ended_at,
   to_date(roll.started_at) AS deploy_date,
   CASE WHEN roll.succeeded THEN 'Succeeded'
        WHEN roll.build_errors > 0 THEN 'Failed'
        ELSE 'In progress' END AS outcome,
   roll.deploy_seconds,
   roll.start_command,
   roll.requirements_installed,
   roll.files_updated,
   CASE WHEN roll.succeeded THEN NULL ELSE roll.failure_reason END AS failure_reason,
   COALESCE(app.app_warnings, 0)  AS app_warnings,
   COALESCE(app.app_errors, 0)    AS app_errors,
   COALESCE(app.http_requests, 0) AS http_requests
FROM roll
LEFT JOIN app ON app.deployment_id = roll.deployment_id
""",
))

# ── v_app_auth_events ────────────────────────────────────────────────────
VIEWS.append(dict(
    name="v_app_auth_events",
    comment=(
        "One row per user sign-in / session event recorded by the Databricks Apps platform (event.name = app.auth). "
        "Use for who is using the app, how often, and whether any logins failed."
    ),
    columns=[
        ("record_id",   "Unique ID of the underlying log record."),
        ("event_time",  "Timestamp (UTC) of the sign-in or session event."),
        ("event_date",  "Calendar date (UTC) of event_time."),
        ("user_email",  "Email of the user who signed in (enduser.id)."),
        ("outcome",     "Result of the auth event, e.g. success or failure."),
        ("reason",      "Why the event happened, e.g. login (new sign-in) or refresh_session (existing session renewed)."),
        ("is_failure",  "True when the outcome is anything other than success."),
    ],
    sql=f"""
SELECT l.record_id,
   l.time                                         AS event_time,
   to_date(l.time)                                AS event_date,
   l.attributes:['enduser.id']::string            AS user_email,
   l.attributes:['app.auth.outcome']::string      AS outcome,
   l.attributes:['app.auth.reason']::string       AS reason,
   COALESCE(l.attributes:['app.auth.outcome']::string, '') <> 'success' AS is_failure
FROM {logs} l
WHERE l.attributes:['event.name']::string = 'app.auth'
""",
))


## Create the Views

In [ ]:
def _q(text):
    return text.replace("'", "\\'")

for v in VIEWS:
    cols = ",\n  ".join(f"`{c}` COMMENT '{_q(d)}'" for c, d in v["columns"])
    ddl = (
        f"CREATE OR REPLACE VIEW {target}.{v['name']} (\n  {cols}\n)\n"
        f"COMMENT '{_q(v['comment'])}'\n"
        f"AS {v['sql'].replace('__TARGET__', target)}"
    )
    spark.sql(ddl)
    print(f"✓ {target}.{v['name']}  ({len(v['columns'])} columns)")

## Validate

Row counts plus a headline summary — the same numbers the Genie Agent will be asked about.

In [ ]:
for v in VIEWS:
    _n = spark.sql(f"SELECT COUNT(*) AS n FROM {target}.{v['name']}").collect()[0]["n"]
    print(f"{v['name']:<24} {_n:>8,} rows")

In [ ]:
display(spark.sql(f"""
    SELECT deployment_id, started_at, outcome, deploy_seconds, start_command,
           failure_reason, app_warnings, app_errors, http_requests
    FROM {target}.v_app_deployments
    ORDER BY started_at DESC
"""))

In [ ]:
display(spark.sql(f"""
    SELECT app_feature,
           COUNT(*)                                AS requests,
           COUNT(DISTINCT user_id)                 AS users,
           COUNT_IF(is_error)                      AS errors,
           ROUND(AVG(approx_duration_seconds), 1)  AS avg_duration_s
    FROM {target}.v_app_http_requests
    GROUP BY app_feature
    ORDER BY requests DESC
"""))

## Summary

Five Genie-ready views now sit next to the raw telemetry tables. Because they are views over the
UC telemetry tables, they are always current — new log lines appear within seconds with no refresh job.